# Search Quality Baseline
Sample issues from DB → generate realistic user queries via Claude → run cosine similarity search → measure Hit@3 and MRR.

In [10]:
%pip install psycopg2-binary python-dotenv openai anthropic numpy -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import psycopg2
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic

load_dotenv()

DATABASE_URL = os.getenv('DATABASE_URL').split('?')[0]
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
anthropic_client = Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

def get_conn():
    return psycopg2.connect(DATABASE_URL, sslmode='require')

def embed(text):
    return openai_client.embeddings.create(model='text-embedding-3-small', input=text[:8000]).data[0].embedding

In [19]:
# Sample N issues that have a non-empty body and resolution_text
N = 100

conn = get_conn()
cur = conn.cursor()
cur.execute("""
    SELECT i.numeric_id, i.issue_number, i.title, i.body, i.resolution_text, l.name as library
    FROM issues i
    JOIN libraries l ON i.library_id = l.id
    WHERE i.body IS NOT NULL AND i.body != ''
      AND i.resolution_text IS NOT NULL AND i.resolution_text != ''
    ORDER BY RANDOM()
    LIMIT %s
""", (N,))
sampled = cur.fetchall()
cur.close()
conn.close()

print(f'Sampled {len(sampled)} issues')
print(f'Example: [{sampled[0][5]}] #{sampled[0][1]} — {sampled[0][2][:80]}')

Sampled 100 issues
Example: [requests] #5438 — Unexpected Prepared Request behavior


In [20]:

# Generate a realistic agent query for each sampled issue using Claude Sonnet 4.6
SYSTEM_CONTEXT = """You are simulating Claude Code — an AI coding agent that helps developers debug and write code.

You have access to an MCP tool called search_bugs. Here is its description:
"Search indexed GitHub issues for bug solutions matching an error message or symptom. Use the exact error string or a short symptom description as the query."

You call this tool when a developer's code throws an error or behaves unexpectedly and you want to find a known solution from a similar GitHub issue. Your query is what you pass to the `query` argument of search_bugs."""

def generate_query(title, body, resolution):
    msg = anthropic_client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=150,
        system=SYSTEM_CONTEXT,
        messages=[{
            'role': 'user',
            'content': f"""Given the following GitHub issue, write the query you would pass to search_bugs if you encountered this bug in a developer's code. Just the query text, nothing else.

Title: {title}
Problem: {body[:500]}
Resolution: {resolution[:300]}"""
        }]
    )
    return msg.content[0].text.strip()

ground_truth = []
for numeric_id, issue_number, title, body, resolution, library in sampled:
    query = generate_query(title, body, resolution)
    ground_truth.append({
        'numeric_id': numeric_id,
        'issue_number': issue_number,
        'library': library,
        'title': title,
        'query': query
    })
    print(f'[{library}] #{issue_number}: {query}')


[requests] #5438: Session.send raises error when headers set to empty dict on PreparedRequest with json POST
[transformers] #1738: BART seq2seq pre-training model implementation
[requests] #3629: `requests post files and json parameters data not sent together`
[psycopg2] #434: `conn.dsn parse connection string attributes host username database`
[langchain] #5978: STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION agent gpt-4 not following through tasks
[requests] #2020: `iter_content chunk_size streaming response not chunked`
[openai-python] #1433: ChatCompletionAssistantMessageParam optional fields pydantic validation error
[flask] #3515: flask stop server after run() called programmatically
[celery] #5750: custom inspect command not found app.control.inspect()
[fastapi] #2430: fastapi change content-type response model JSONResponse default_response_class
[psycopg2] #1767: psycopg2 PostgreSQL system administration functions no FROM clause error
[next.js] #8834: splitChunks configuration brea

In [25]:
def to_vec(embedding):
    return "[" + ",".join(str(x) for x in embedding) + "]"

def search(query_text, target_id, top_k=3):
    q_emb = to_vec(embed(query_text))
    conn = get_conn()
    cur = conn.cursor()

    cur.execute("""
        SELECT issue_id, 1 - (embedding <=> %s::vector) AS similarity
        FROM embeddings
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """, (q_emb, q_emb, top_k))
    top_rows = cur.fetchall()

    cur.execute("""
        SELECT rank FROM (
            SELECT issue_id, ROW_NUMBER() OVER (ORDER BY embedding <=> %s::vector) AS rank
            FROM embeddings
        ) ranked WHERE issue_id = %s
    """, (q_emb, target_id))
    rank_row = cur.fetchone()
    full_rank = rank_row[0] if rank_row else None

    cur.close()
    conn.close()
    return [iid for iid, _ in top_rows], full_rank


results = []
for item in ground_truth:
    top3, full_rank = search(item['query'], item['numeric_id'])
    rank = next((i+1 for i, iid in enumerate(top3) if iid == item['numeric_id']), None)
    results.append({**item, 'top3': top3, 'hit': rank is not None, 'rank_in_top3': rank, 'actual_rank': full_rank})
    status = f'HIT @{rank}' if rank else f'MISS (rank {full_rank})'
    print(f'[{item["library"]}] #{item["issue_number"]}: {status}')

[requests] #5438: HIT @1
[transformers] #1738: HIT @1
[requests] #3629: HIT @1
[psycopg2] #434: HIT @1
[langchain] #5978: HIT @1
[requests] #2020: MISS (rank 46)
[openai-python] #1433: HIT @1
[flask] #3515: HIT @1
[celery] #5750: HIT @1
[fastapi] #2430: HIT @3
[psycopg2] #1767: HIT @1
[next.js] #8834: HIT @1
[celery] #8751: HIT @1
[next.js] #178: HIT @1
[celery] #6682: HIT @1
[requests] #6408: HIT @1
[flask] #4813: HIT @1
[transformers] #7920: HIT @1
[pydantic] #3579: HIT @1
[pydantic] #4096: HIT @3
[pyrefly] #996: MISS (rank 16)
[pydantic] #6663: MISS (rank 5)
[pydantic] #1650: HIT @1
[pydantic] #5521: HIT @1
[requests] #6271: HIT @1
[pytest] #2368: HIT @1
[next.js] #2718: HIT @1
[supabase-py] #478: HIT @1
[transformers] #5321: HIT @1
[transformers] #9745: MISS (rank 28217)
[fastapi] #5281: HIT @3
[langchain] #9212: HIT @2
[psycopg2] #278: HIT @1
[celery] #3376: HIT @2
[pydantic] #3173: HIT @1
[pyrefly] #2382: MISS (rank 5)
[langchain] #7129: MISS (rank 7)
[python-dotenv] #169: HIT @1

In [26]:
# Metrics
hit_at_3 = sum(1 for r in results if r['hit']) / len(results)
mrr = sum(1 / r['actual_rank'] for r in results if r['actual_rank']) / len(results)

print(f'N = {len(results)}')
print(f'Hit@3:  {hit_at_3:.2%}')
print(f'MRR:    {mrr:.4f}')

misses = [r for r in results if not r['hit']]
print(f'\nMisses ({len(misses)}):')
for r in misses:
    print(f'  [{r["library"]}] #{r["issue_number"]} rank={r["actual_rank"]} — query: {r["query"]}')
    print(f'    title: {r["title"][:80]}')

N = 100
Hit@3:  81.00%
MRR:    0.7293

Misses (19):
  [requests] #2020 rank=46 — query: `iter_content chunk_size streaming response not chunked`
    title: Iterating over chunks that are not terminated by new lines
  [pyrefly] #996 rank=16 — query: duplicate function type signatures union display deduplication
    title: Merge function types for display when they have the same type signature
  [pydantic] #6663 rank=5 — query: Json[type] schema generated returns string rather than type inside Json[]
    title: `Json[type]` schema generated now returns `string` rather than `type`
  [transformers] #9745 rank=28217 — query: `TypeError: forward() got an unexpected keyword argument 'head_mask' EncoderDecoderModel longformer`
    title: fine tune patrickvonplaten/longformer2roberta-cnn_dailymail-fp16 using LED updat
  [pyrefly] #2382 rank=5 — query: walrus operator ternary expression unbound name false branch
    title: unbound name when using walrus operator within a ternary expression
  [la

In [27]:
from pydantic import BaseModel
from typing import List

class RankingResult(BaseModel):
    top_3_indices: List[int]  # 1-based indices into the candidates list, in order of relevance

def rerank(query, candidates):
    numbered = "\n".join([
        f"{i+1}. [{c['library']}] #{c['issue_number']} (distance: {c['distance']:.4f})\n   Title: {c['title']}\n   Body: {c['body']}"
        for i, c in enumerate(candidates)
    ])
    completion = openai_client.chat.completions.parse(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": "You are a relevance judge for a bug solutions search engine. Given a developer query and a list of GitHub issues, return the 1-based indices of the 3 most relevant issues in order of relevance."
            },
            {
                "role": "user",
                "content": f"Query: {query}\n\nCandidates:\n{numbered}"
            }
        ],
        response_format=RankingResult,
    )
    picks = completion.choices[0].message.parsed.top_3_indices
    return [candidates[i-1]['numeric_id'] for i in picks if 0 < i <= len(candidates)]


def search_reranked(query_text, target_id, top_k=10):
    q_emb = to_vec(embed(query_text))
    conn = get_conn()
    cur = conn.cursor()

    cur.execute("""
        SELECT e.issue_id, i.issue_number, i.title, l.name,
               1 - (e.embedding <=> %s::vector) AS similarity,
               i.body
        FROM embeddings e
        JOIN issues i ON e.issue_id = i.numeric_id
        JOIN libraries l ON i.library_id = l.id
        ORDER BY e.embedding <=> %s::vector
        LIMIT %s
    """, (q_emb, q_emb, top_k))
    top_rows = cur.fetchall()

    cur.execute("""
        SELECT rank FROM (
            SELECT issue_id, ROW_NUMBER() OVER (ORDER BY embedding <=> %s::vector) AS rank
            FROM embeddings
        ) ranked WHERE issue_id = %s
    """, (q_emb, target_id))
    rank_row = cur.fetchone()
    full_rank = rank_row[0] if rank_row else None

    cur.close()
    conn.close()

    candidates = [
        {
            'numeric_id': r[0],
            'issue_number': r[1],
            'title': r[2],
            'library': r[3],
            'distance': round(1 - r[4], 4),
            'body': (r[5] or '')[:300],
        }
        for r in top_rows
    ]
    reranked_ids = rerank(query_text, candidates)
    return reranked_ids, full_rank


reranked_results = []
for item in ground_truth:
    top3_reranked, full_rank = search_reranked(item['query'], item['numeric_id'])
    rank = next((i+1 for i, iid in enumerate(top3_reranked) if iid == item['numeric_id']), None)
    reranked_results.append({**item, 'hit': rank is not None, 'rank_in_top3': rank, 'actual_rank': full_rank})
    status = f'HIT @{rank}' if rank else f'MISS (rank {full_rank})'
    print(f'[{item["library"]}] #{item["issue_number"]}: {status}')

[requests] #5438: HIT @1
[transformers] #1738: HIT @1
[requests] #3629: HIT @1
[psycopg2] #434: HIT @1
[langchain] #5978: HIT @1
[requests] #2020: MISS (rank 46)
[openai-python] #1433: HIT @1
[flask] #3515: HIT @1
[celery] #5750: HIT @1
[fastapi] #2430: HIT @1
[psycopg2] #1767: HIT @1
[next.js] #8834: HIT @1
[celery] #8751: HIT @1
[next.js] #178: HIT @1
[celery] #6682: HIT @1
[requests] #6408: HIT @1
[flask] #4813: HIT @1
[transformers] #7920: HIT @1
[pydantic] #3579: HIT @1
[pydantic] #4096: MISS (rank 3)
[pyrefly] #996: MISS (rank 16)
[pydantic] #6663: HIT @1
[pydantic] #1650: HIT @1
[pydantic] #5521: HIT @1
[requests] #6271: HIT @1
[pytest] #2368: HIT @1
[next.js] #2718: HIT @1
[supabase-py] #478: HIT @1
[transformers] #5321: HIT @1
[transformers] #9745: MISS (rank 28217)
[fastapi] #5281: HIT @1
[langchain] #9212: HIT @1
[psycopg2] #278: HIT @1
[celery] #3376: HIT @1
[pydantic] #3173: HIT @1
[pyrefly] #2382: HIT @1
[langchain] #7129: HIT @1
[python-dotenv] #169: HIT @1
[pydantic] #2

In [28]:
# Compare baseline vs reranked
hit_at_3_reranked = sum(1 for r in reranked_results if r['hit']) / len(reranked_results)
mrr_reranked = sum(1 / r['actual_rank'] for r in reranked_results if r['actual_rank']) / len(reranked_results)

print(f"{'':20} {'Baseline':>10} {'Reranked':>10}")
print(f"{'Hit@3':20} {hit_at_3:>10.2%} {hit_at_3_reranked:>10.2%}")
print(f"{'MRR':20} {mrr:>10.4f} {mrr_reranked:>10.4f}")

misses = [r for r in reranked_results if not r['hit']]
print(f'\nRemaining misses ({len(misses)}):')
for r in misses:
    print(f'  [{r["library"]}] #{r["issue_number"]} rank={r["actual_rank"]} — {r["query"]}')

                       Baseline   Reranked
Hit@3                    81.00%     86.00%
MRR                      0.7293     0.7293

Remaining misses (14):
  [requests] #2020 rank=46 — `iter_content chunk_size streaming response not chunked`
  [pydantic] #4096 rank=3 — mypy does not detect attribute access error in pydantic model
  [pyrefly] #996 rank=16 — duplicate function type signatures union display deduplication
  [transformers] #9745 rank=28217 — `TypeError: forward() got an unexpected keyword argument 'head_mask' EncoderDecoderModel longformer`
  [next.js] #3323 rank=296 — AssertionError Invalid register options "value" must be an object
  [celery] #7001 rank=22 — beat_schedule task not executing scheduled tasks not running
  [fastapi] #836 rank=24 — recursive nested pydantic models KeyError documentation
  [requests] #4984 rank=161 — requests.exceptions.InvalidProxyURL Please check proxy URL
  [redis-py] #2218 rank=16 — Lock.acquire() TypeError non-blocking positional argument re